<a href="https://colab.research.google.com/github/gratefulgee/fcc-ML-NN-sms-text-classifier/blob/main/Copy_of_fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries
!pip install tensorflow
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


2.21.0


In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2026-03-25 00:51:55--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv.1’

train-data.tsv.1    100%[===================>] 349.84K  --.-KB/s    in 0.01s   

2026-03-25 00:51:55 (25.9 MB/s) - ‘train-data.tsv.1’ saved [358233/358233]

--2026-03-25 00:51:55--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv.1’

valid-data.tsv.1    100%[====

In [3]:
train = pd.read_csv("train-data.tsv", sep="\t", header=None, names=["label", "message"])
test  = pd.read_csv("valid-data.tsv", sep="\t", header=None, names=["label", "message"])
train.head(100)

,label,message
0,ham,ahhhh...just woken up!had a bad dream about u ...
1,ham,you can never do nothing
2,ham,"now u sound like manky scouse boy steve,like! ..."
3,ham,mum say we wan to go then go... then she can s...
4,ham,never y lei... i v lazy... got wat? dat day ü ...
...,...,...
95,ham,ok... i din get ur msg...
96,spam,"free msg. sorry, a service you ordered from 81..."
97,ham,:-( that's not v romantic!
98,ham,probably gonna swing by in a wee bit


In [4]:
# convert label
train["label_n"] = train["label"].apply(lambda x: 1 if x=="spam" else 0)
test["label_n"] = test["label"].apply(lambda x: 1 if x=="spam" else 0)

#convert to numpy array
train_text = train["message"].astype(str).values
train_label = train["label_n"].values

test_text = test["message"].astype(str).values
test_label = test["label_n"].values
train_text

array(['ahhhh...just woken up!had a bad dream about u tho,so i dont like u right now :) i didnt know anything about comedy night but i guess im up for it.',
       'you can never do nothing',
       'now u sound like manky scouse boy steve,like! i is travelling on da bus home.wot has u inmind 4 recreation dis eve?',
       ...,
       'free entry into our £250 weekly competition just text the word win to 80086 now. 18 t&c www.txttowin.co.uk',
       '-pls stop bootydelious (32/f) is inviting you to be her friend. reply yes-434 or no-434 see her: www.sms.ac/u/bootydelious stop? send stop frnd to 62468',
       "tell my  bad character which u dnt lik in me. i'll try to change in  &lt;#&gt; . i ll add tat 2 my new year resolution. waiting for ur reply.be frank...good morning."],
      dtype=object)

In [5]:
# Tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_text)

# convert to sequence
train_seq = tokenizer.texts_to_sequences(train_text)
test_seq = tokenizer.texts_to_sequences(test_text)

# padding
max_len = 100
train_pad = tf.keras.preprocessing.sequence.pad_sequences(train_seq, maxlen=max_len, padding='post')
test_pad = tf.keras.preprocessing.sequence.pad_sequences(test_seq, maxlen=max_len, padding='post')


In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(20000, 32, input_length=max_len),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(train_pad, train_label, epochs=5, validation_split=0.2)

loss, acc = model.evaluate(test_pad, test_label)
print("ACC =", acc)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.8552 - loss: 0.3409 - val_accuracy: 0.9629 - val_loss: 0.1660
Epoch 2/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.9821 - loss: 0.0847 - val_accuracy: 0.9821 - val_loss: 0.0671
Epoch 3/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.9934 - loss: 0.0298 - val_accuracy: 0.9809 - val_loss: 0.0599
Epoch 4/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.9982 - loss: 0.0103 - val_accuracy: 0.9797 - val_loss: 0.0688
Epoch 5/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.9994 - loss: 0.0041 - val_accuracy: 0.9844 - val_loss: 0.0615
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9871 - loss: 0.0622
ACC = 0.9870689511299133


In [7]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    seq = tokenizer.texts_to_sequences([pred_text])
    pad = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=max_len, padding='post')
    prob = model.predict(pad)[0][0]
    label = 'spam' if prob > 0.5 else 'ham'
    return [float(prob), label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step
[0.0007651476189494133, 'ham']


In [8]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
You haven't passed yet. Keep trying.
